# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kratosontren/flyrank-ml-work/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from datasets import load_dataset
from itertools import islice
import pandas as pd

# Load FlyRank warehouse sample
daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

# Use the first 5000 rows
sample = pd.DataFrame(list(islice(daily, 5000)))

print("Dataset Shape:", sample.shape)
sample.head()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Dataset Shape: (5000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


# Research Question

## Title

**Content Refresh Prioritization Using Historical Search Performance Signals: A Decision-Support Approach**

## Abstract

This project investigates whether historical search performance signals can be used to prioritize pages for manual content review. A Logistic Regression model was compared against a transparent rule-based baseline using the same historical dataset and evaluation design. The study used historical Google Search Console metrics and a proxy target derived from search position to evaluate content refresh prioritization. Model performance was evaluated using both random and grouped validation splits to assess generalization while avoiding information leakage. The results are intended as decision-support for SEO and content teams rather than proof that refreshing content will improve search performance.

## Research Question

Can historical search performance signals be used to prioritize pages for manual content review?

## Decision Supported

The output supports SEO analysts and content teams in deciding which pages should be reviewed first for possible optimization.

# Data

The analysis uses the FlyRank ML Internship dataset.

Dataset release:
FlyRank Internship Warehouse

Primary table:
fact_content_daily_performance

Primary historical signals:

- Google Search Console impressions
- Google Search Console clicks
- Average search position

The analysis uses historical observations only.

Excluded:

- Future outcome windows
- Label-derived variables
- Internal product recommendation flags
- Client-identifying information beyond anonymized identifiers

The dataset was used only for educational and research purposes, and no client-identifying information appears in this work.

In [ ]:
sample[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
].describe()

,gsc_impressions,gsc_clicks,gsc_avg_position
count,5000.000000,5000.000000,5000.000000
mean,11.489400,0.105600,25.718199
std,18.728258,0.421526,23.504413
min,1.000000,0.000000,0.000000
25%,2.000000,0.000000,7.833333
50%,6.000000,0.000000,16.445906
75%,14.000000,0.000000,37.200000
max,424.000000,8.000000,127.000000


# Methodology

The study compares a transparent baseline rule with a Logistic Regression model.

Features:

- gsc_impressions
- gsc_clicks

Proxy target:

Pages with poorer average search position than the historical median.

Baseline:

Simple priority score based on historical search signals.

Validation:

- Random train-test split
- Grouped validation using client identifiers

Leakage prevention:

- No future outcome variables
- No label-derived features
- The feature used to create the proxy target was excluded from model training

The objective is to evaluate decision-support performance rather than maximize predictive accuracy.

In [ ]:
feature_summary = pd.DataFrame({

    "Feature":[
        "gsc_impressions",
        "gsc_clicks"
    ],

    "Used":[
        True,
        True
    ]

})

feature_summary

,Feature,Used
0,gsc_impressions,True
1,gsc_clicks,True


# Results

The Logistic Regression model was compared against the Week-4 baseline using the same proxy target and evaluation strategy.

Grouped validation produced lower performance than the random split, suggesting that the grouped split provides a more realistic estimate of model generalization.

These results should be interpreted as measured performance on historical data rather than production performance.

# Limitations

This study has several limitations.

- The proxy target is an approximation of refresh need.
- The dataset contains sparse click observations.
- The analysis is observational and cannot establish causal relationships.
- Historical search behaviour may change over time.
- The recommendations require human review before implementation.

The results should therefore be interpreted as decision-support rather than proof of future search improvements.

# Ranked Recommendations

Pages with higher priority scores should be reviewed first.

Recommended actions include:

- Refresh content with poor ranking performance.
- Improve title tags and meta descriptions where click opportunities exist.
- Monitor moderate-performing pages before taking action.
- Leave low-priority pages unchanged unless additional evidence becomes available.

These recommendations are intended to guide human reviewers rather than automate publishing decisions.

In [ ]:
import pandas as pd

# Create a working copy
queue = sample.copy()

# Same priority score used in ML-10
queue["priority_score"] = (
    queue["gsc_impressions"] / 10
    - queue["gsc_clicks"]
)

# Reason codes
def reason_code(score):
    if score >= queue["priority_score"].quantile(0.75):
        return "RC-01"
    elif score >= queue["priority_score"].median():
        return "RC-02"
    elif score >= queue["priority_score"].quantile(0.25):
        return "RC-03"
    else:
        return "RC-04"

queue["reason_code"] = queue["priority_score"].apply(reason_code)

# Recommended actions
action_map = {
    "RC-01": "Refresh content",
    "RC-02": "Improve CTR elements",
    "RC-03": "Monitor performance",
    "RC-04": "No immediate action"
}

queue["recommended_action"] = queue["reason_code"].map(action_map)

# Final ranked queue
ranked_queue = queue.sort_values(
    "priority_score",
    ascending=False
)

print("Ranked queue created successfully.")
ranked_queue.head()

Ranked queue created successfully.


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,priority_score,reason_code,recommended_action
3825,2025-02-11,client_9958f0a7ae1df715,content_213eb91f21a43550,True,True,True,False,424,0,466,...,0,0,0,0,0,0,0,42.4,RC-01,Refresh content
446,2025-01-28,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,303,3,549,...,0,0,0,0,0,0,0,27.3,RC-01,Refresh content
1221,2025-01-31,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,305,5,484,...,0,0,0,0,0,0,0,25.5,RC-01,Refresh content
146,2025-01-27,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,266,6,482,...,0,0,0,0,0,0,0,20.6,RC-01,Refresh content
3884,2025-02-11,client_9958f0a7ae1df715,content_f94fe855380e150f,True,True,True,False,227,3,378,...,0,0,0,0,0,0,0,19.7,RC-01,Refresh content


In [ ]:
ranked_queue[
    [
        "priority_score",
        "reason_code",
        "recommended_action"
    ]
].head(20)

,priority_score,reason_code,recommended_action
3825,42.4,RC-01,Refresh content
446,27.3,RC-01,Refresh content
1221,25.5,RC-01,Refresh content
146,20.6,RC-01,Refresh content
3884,19.7,RC-01,Refresh content
766,19.0,RC-01,Refresh content
1019,18.7,RC-01,Refresh content
1537,16.5,RC-01,Refresh content
349,16.4,RC-01,Refresh content
953,16.2,RC-01,Refresh content


# Artifacts

The deployed research paper includes:

- Model vs baseline comparison table
- Ranked recommendation queue
- Priority score distribution
- Feature summary
- Leakage audit summary
- Monitoring recommendations

These artifacts are generated directly from this notebook to support reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Build target
threshold = sample["gsc_avg_position"].median()

sample["target"] = (
    sample["gsc_avg_position"] > threshold
).astype(int)

# Features
X = sample[
    [
        "gsc_impressions",
        "gsc_clicks"
    ]
].fillna(0)

y = sample["target"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Logistic Regression
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

model_pred = model.predict(X_test)

# Baseline
baseline_score = (
    X_test["gsc_impressions"]/10
    -
    X_test["gsc_clicks"]
)

baseline_pred = (
    baseline_score >
    baseline_score.median()
).astype(int)

# Results table
results = pd.DataFrame({

    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ],

    "Baseline":[

        accuracy_score(y_test,baseline_pred),

        precision_score(
            y_test,
            baseline_pred,
            zero_division=0
        ),

        recall_score(
            y_test,
            baseline_pred,
            zero_division=0
        ),

        f1_score(
            y_test,
            baseline_pred,
            zero_division=0
        )

    ],

    "Logistic Regression":[

        accuracy_score(y_test,model_pred),

        precision_score(
            y_test,
            model_pred,
            zero_division=0
        ),

        recall_score(
            y_test,
            model_pred,
            zero_division=0
        ),

        f1_score(
            y_test,
            model_pred,
            zero_division=0
        )

    ]

})

results

,Metric,Baseline,Logistic Regression
0,Accuracy,0.511000,0.552000
1,Precision,0.511156,0.528261
2,Recall,0.504000,0.972000
3,F1,0.507553,0.684507


In [ ]:
import os

os.makedirs("work/figures", exist_ok=True)

results.to_csv(
    "work/figures/model_results.csv",
    index=False
)

ranked_queue.head(20).to_csv(
    "work/figures/top20_recommendations.csv",
    index=False
)

print("Artifacts exported.")

Artifacts exported.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Committed to my repo under `work/notebooks/`.
- [x] My deployed paper will include all required sections, including the Abstract and Acknowledgments.
- [x] ML-12 deliverables are included below.

# 5-Minute Demo Outline

1. Research question
2. Dataset overview
3. Baseline rule
4. Logistic Regression model
5. Validation strategy
6. Results comparison
7. Recommendations
8. Limitations
9. Future work

# Social Post

Completed the FlyRank ML Internship capstone on content refresh prioritization using historical search performance data. The project compares a transparent baseline with Logistic Regression, includes grouped validation, leakage checks, and a practical content action playbook. The work emphasizes honest evaluation and decision-support over overly optimistic claims.

# Employer Summary

Built a complete machine learning workflow using anonymized production-scale search data. Developed baseline rules, validated machine learning models, performed leakage audits, and translated model outputs into practical content recommendations while emphasizing reproducibility and honest evaluation.

# ML-12 — 5 Minute Demo Outline

## 1. Introduction

Hello everyone.

My project investigates whether historical search performance signals can help prioritize pages for manual content review.

The objective is to support SEO analysts by ranking pages that may benefit from a content refresh.

---

## 2. Dataset

The project uses the FlyRank ML Internship Warehouse dataset containing anonymized historical search performance observations.

Only historical Google Search Console features available before prediction time were used.

Future information and label-derived features were excluded to prevent information leakage.

---

## 3. Method

I first created a transparent rule-based baseline to prioritize pages.

I then trained a Logistic Regression model using historical impressions and clicks.

The model was evaluated using both random and grouped validation splits to compare generalization performance.

---

## 4. Results

The grouped validation produced lower performance than the random split, demonstrating that random evaluation provides optimistic estimates.

This reinforced the importance of honest validation and leakage prevention.

The final output is a ranked recommendation queue for manual content review.

---

## 5. Recommendation

The model is intended to support—not replace—human decision making.

Editors can use the ranked queue to prioritize review while considering business context, content quality, and search intent.

The recommendations are decision-support rather than automated publishing decisions.

---

## Closing

This project demonstrates an end-to-end machine learning workflow including problem framing, data contracts, feature engineering, baseline comparison, validation, leakage auditing, and actionable recommendations using production-style search data.

# Social Post

I recently completed an end-to-end machine learning capstone as part of the FlyRank ML Internship.

The project explored how historical search performance signals can support content refresh prioritization using interpretable machine learning. The workflow included data contracts, leakage prevention, baseline comparison, grouped validation, and a practical action playbook built on anonymized production-style search data.

Repository:
https://github.com/kratosontren/flyrank-ml-work

# Employer Summary

I built an end-to-end machine learning workflow using anonymized production-style search data from the FlyRank ML Internship Warehouse.

The project compared a transparent rule-based baseline with a Logistic Regression model for prioritizing pages for manual content review while applying honest validation and leakage checks.

The final system produces a ranked recommendation queue intended to support SEO analysts with interpretable, decision-support recommendations.